# Homework 1 — Negotiation Exercise


Multiagent Learning | Agentic AI (GPT edition)

An **LLM agent** is the classic agent loop with a pretrained model as the policy.  
A **society** is agents + shared state + a protocol.

This notebook:
1. Baseline Buyer / Seller / Mediator negotiation
2. Exercise 1 — second seller with a higher floor
3. Exercise 2 — buyer memory across several negotiations

Model: `gpt-5.4-mini` via the OpenAI API.

## 1. Setup

Requires `OPENAI_API_KEY` in the environment or in a local `.env` file.  

In [4]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Load .env from the notebook folder (and CWD as fallback)
for candidate in [Path.cwd() / ".env", Path.cwd().parent / ".env"]:
    load_dotenv(candidate)
load_dotenv()

API_KEY = os.environ.get("OPENAI_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Copy .env.example to .env and add your key, "
        "or set $env:OPENAI_API_KEY in PowerShell."
    )

MODEL = "gpt-5.4-mini"
client = OpenAI(api_key=API_KEY)
print(f"Ready. Model = {MODEL}")

Ready. Model = gpt-5.4-mini


## 2. Shared primitives

- **Agent** — role prompt (policy) + view of the board (percept) + one model call (action)
- **Orchestrator** — round-robin turn-taking; stop on `DEAL:` or max turns
- **Board** — shared list of `(speaker, text)`; the only shared state

In [5]:
class Agent:
    """LLM-wrapper agent: system prompt is identity + goal; board is the percept."""

    def __init__(self, name: str, role: str, memory: str | None = None):
        self.name = name
        self.role = role
        self.memory = memory  # optional cross-negotiation memory (Exercise 2)

    def _system(self) -> str:
        parts = [
            f"You are {self.name}. {self.role}",
            "Reply with your next line only, 1-2 sentences.",
        ]
        if self.memory:
            parts.append(
                "Memory of past negotiations (use to bargain smarter; "
                "do not invent deals that are not on the current board):\n"
                + self.memory
            )
        return " ".join(parts[:2]) + ("\n\n" + parts[2] if len(parts) > 2 else "")

    def act(self, board: list[tuple[str, str]]) -> str:
        view = "\n".join(f"{who}: {text}" for who, text in board)
        r = client.chat.completions.create(
            model=MODEL,
            max_completion_tokens=200,
            messages=[
                {"role": "system", "content": self._system()},
                {"role": "user", "content": view or "Begin."},
            ],
        )
        return (r.choices[0].message.content or "").strip()


def extract_deal_price(board: list[tuple[str, str]]) -> float | None:
    """Parse the first DEAL: <price> found on the board."""
    for _, text in board:
        m = re.search(r"DEAL:\s*([0-9]+(?:\.[0-9]+)?)", text, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return None


def run_negotiation(
    agents: list[Agent],
    max_turns: int = 15,
    verbose: bool = True,
) -> dict:
    """Round-robin orchestrator. Returns board, turns used, deal price (or None)."""
    board: list[tuple[str, str]] = []
    for turn in range(max_turns):
        speaker = agents[turn % len(agents)]
        line = speaker.act(board)
        board.append((speaker.name, line))
        if verbose:
            print(f"{speaker.name}: {line}")
        # Lecture stop condition: DEAL: appears in the spoken line
        if re.search(r"DEAL:\s*", line, flags=re.IGNORECASE):
            break
    price = extract_deal_price(board)
    return {
        "board": board,
        "turns": len(board),
        "deal_price": price,
        "deal": price is not None,
    }


def summarize_board(board: list[tuple[str, str]], deal_price: float | None) -> str:
    """Short text summary for buyer memory (Exercise 2)."""
    outcome = f"DEAL at {deal_price:.0f} EUR" if deal_price is not None else "no deal"
    snippet = " | ".join(f"{w}: {t[:80]}" for w, t in board[-4:])
    return f"Outcome: {outcome}. Last lines: {snippet}"


print("Primitives loaded.")

Primitives loaded.


## 3. Baseline — three agents, one product

| Agent | Constraint |
|-------|------------|
| Buyer | max **600** EUR (hidden) |
| Seller | min **450** EUR, opens at **800** |
| Mediator | neutral; declares `DEAL: <price>` when they agree |

Zone of possible agreement: **450–600 EUR**.

In [6]:
baseline_agents = [
    Agent(
        "Buyer",
        "You want a used laptop and will pay at most 600 EUR. Never reveal your limit.",
    ),
    Agent(
        "Seller",
        "You sell a used laptop. You will not go below 450 EUR. Open at 800 EUR.",
    ),
    Agent(
        "Mediator",
        "You are neutral. When buyer and seller agree on a price, reply 'DEAL: <price>'. "
        "Otherwise nudge them toward agreement.",
    ),
]

print("=== Baseline negotiation ===")
baseline = run_negotiation(baseline_agents, max_turns=9)
print(
    f"\nTurns: {baseline['turns']} | Deal: {baseline['deal']} | "
    f"Price: {baseline['deal_price']}"
)

=== Baseline negotiation ===
Buyer: Hi, I’m looking for a used laptop in good condition—what models do you have available?
Seller: Hi, I have a well-maintained laptop available in good condition, and I’m asking 800 EUR for it. If you’re interested, I can share the exact model and specs.
Mediator: Could you share the exact model and specs so the buyer can compare value, and if it fits their needs maybe you can also mention whether 800 EUR is firm or negotiable?
Buyer: Yes, please share the exact model and specs, and let me know whether the price is flexible.
Seller: It’s a Lenovo ThinkPad T14 with an Intel Core i7, 16GB RAM, and 512GB SSD, in very good condition. The price is 800 EUR, and I can be a little flexible, but not below 450 EUR.
Mediator: Could you tell the buyer how much flexibility you have within that range, and buyer, what’s your target budget for this ThinkPad?
Buyer: Thanks for the details — the ThinkPad T14 sounds interesting, but 800 EUR is a bit high for me. Would you

### Baseline notes 

- First started with 9 turns but this was not enough to settle
- Seller starts with 800 after talking about the specs of the laptop
- Mediator is doing its job, but a lot of initial talk about specs, might need to delete it
- Moved to limit 15 and that made it easier for them to understand, Mediator might be getting the job done too fast


## 4. Exercise 1 — add a second seller

**Task:** Add another seller with a **higher bottom price** for the same used laptop. Describe the effect in your own words.

Here: **Seller2** will not go below **520** EUR (still must compete with Seller1's floor of 450).

In [7]:
ex1_agents = [
    Agent(
        "Buyer",
        "You want a used laptop and will pay at most 600 EUR. Never reveal your limit. "
        "There are two sellers; prefer the better (lower) price.",
    ),
    Agent(
        "Seller1",
        "You sell a used laptop. You will not go below 450 EUR. Open at 800 EUR. "
        "You compete with Seller2 for the same buyer.",
    ),
    Agent(
        "Seller2",
        "You sell the same used laptop model. You will not go below 520 EUR. Open at 780 EUR. "
        "You compete with Seller1 for the same buyer.",
    ),
    Agent(
        "Mediator",
        "You are neutral. When the buyer and one seller clearly agree on a price, "
        "reply 'DEAL: <price> with <SellerName>'. Otherwise nudge them.",
    ),
]

print("=== Exercise 1: two sellers ===")
ex1 = run_negotiation(ex1_agents, max_turns=15)
print(
    f"\nTurns: {ex1['turns']} | Deal: {ex1['deal']} | Price: {ex1['deal_price']}"
)

=== Exercise 1: two sellers ===
Buyer: Hi, I’m looking for a used laptop—what models do you have and what’s the condition?
Seller1: Hi! I have a used Dell XPS 13, 11th gen i7, 16GB RAM, 512GB SSD, in very good condition with normal light wear and fully working battery/keyboard/screen. I’m asking 800 EUR.
Seller2: Hi, I have the same Dell XPS 13, 11th gen i7, 16GB RAM, 512GB SSD, in very good condition with normal light wear and fully working battery, keyboard, and screen. My asking price is 780 EUR.
Mediator: Could the buyer share their target budget, and would either seller be willing to move a bit on price?
Buyer: Thanks — both look good, and I’d be interested in the Dell XPS 13. Could either of you do a better price?
Seller1: I can move a bit, but I’m still at 800 EUR for now given the condition and specs. If you’re serious, I can work with you on a small discount, but I can’t go below 450 EUR.
Seller2: I can come down a little, but I’m still firm at 780 EUR for this Dell XPS 13 giv

### Exercise 1 — your analysis

- Mediator is again doing a good job, seller 1 nad 2 are pretty much having the same answers but with different minimum price
- Need to raise the max turns to 15-20

- Seller 2 is able to get the deal faster as it undercuts the sale by a bit, maybe 10-20 euros, but that is enough to convince the Buyer to go for him
- I am surprised Seller 2 is able to go bellow even though its limit is higher, I suspect that is because it is able to read Sellers 1 offer and undercut it and the Buyer accepts it without leaving time for Seller 1 to counterofffer because if that would be introduced, Seller 1 will win because he can go lower

## 5. Exercise 2 — buyer memory across negotiations

**Task:** Give the Buyer a memory of past negotiations and run several iterations. Comment on the results.

Each iteration resets the **board**, but the Buyer's **memory** grows with short summaries of previous outcomes.

In [8]:
N_ITERATIONS = 3
buyer_memory_lines: list[str] = []
ex2_results: list[dict] = []

for i in range(1, N_ITERATIONS + 1):
    memory_text = "\n".join(f"- Round {j}: {s}" for j, s in enumerate(buyer_memory_lines, 1))
    buyer = Agent(
        "Buyer",
        "You want a used laptop and will pay at most 600 EUR. Never reveal your limit. "
        "Use past outcomes to open lower or concede slower when it helped before.",
        memory=memory_text or None,
    )
    seller = Agent(
        "Seller",
        "You sell a used laptop. You will not go below 450 EUR. Open at 800 EUR.",
    )
    mediator = Agent(
        "Mediator",
        "You are neutral. When buyer and seller agree on a price, reply 'DEAL: <price>'. "
        "Otherwise nudge them toward agreement.",
    )

    print(f"\n=== Exercise 2 — iteration {i}/{N_ITERATIONS} ===")
    if memory_text:
        print("(Buyer has memory of previous rounds)")
    result = run_negotiation([buyer, seller, mediator], max_turns=15)
    summary = summarize_board(result["board"], result["deal_price"])
    buyer_memory_lines.append(summary)
    ex2_results.append(
        {
            "iteration": i,
            "turns": result["turns"],
            "deal": result["deal"],
            "deal_price": result["deal_price"],
            "summary": summary,
        }
    )
    print(f"Summary stored: {summary}")


=== Exercise 2 — iteration 1/3 ===
Buyer: Hi, I’m interested in the laptop. What’s the best price you can do?
Seller: Hi, thanks for your interest — I can do 800 EUR for the laptop.
Mediator: What’s the highest you’d be comfortable paying, and could the seller shave a bit off 800 EUR if you can close today?
Buyer: Could you do 650 EUR if I can come pick it up and close today?
Seller: Thanks for the offer, but 650 EUR is too low for me. I can come down to 750 EUR if you can pick it up and close today.
Mediator: Could the buyer meet at 725 EUR for a same-day pickup?
Buyer: Would you consider 700 EUR for same-day pickup?
Seller: I can meet you at 725 EUR for same-day pickup.
Mediator: DEAL: 725 EUR
Summary stored: Outcome: DEAL at 725 EUR. Last lines: Mediator: Could the buyer meet at 725 EUR for a same-day pickup? | Buyer: Would you consider 700 EUR for same-day pickup? | Seller: I can meet you at 725 EUR for same-day pickup. | Mediator: DEAL: 725 EUR

=== Exercise 2 — iteration 2/3 ===

In [9]:
# Summary table across iterations
print(f"{'iter':>4}  {'turns':>5}  {'deal':>5}  {'price':>8}")
print("-" * 30)
for row in ex2_results:
    price = row["deal_price"] if row["deal_price"] is not None else "—"
    print(f"{row['iteration']:>4}  {row['turns']:>5}  {str(row['deal']):>5}  {str(price):>8}")

print("\nBuyer memory at the end:")
for j, s in enumerate(buyer_memory_lines, 1):
    print(f"  {j}. {s}")

iter  turns   deal     price
------------------------------
   1      9   True     725.0
   2      6   True     650.0
   3     12   True     640.0

Buyer memory at the end:
  1. Outcome: DEAL at 725 EUR. Last lines: Mediator: Could the buyer meet at 725 EUR for a same-day pickup? | Buyer: Would you consider 700 EUR for same-day pickup? | Seller: I can meet you at 725 EUR for same-day pickup. | Mediator: DEAL: 725 EUR
  2. Outcome: DEAL at 650 EUR. Last lines: Mediator: Could you meet in the middle at 600 EUR for same-day pickup, or is 650 EUR the l | Buyer: Would you consider 600 EUR for same-day pickup? | Seller: I can do 650 EUR for same-day pickup, but I can’t go down to 600 EUR. If that wo | Mediator: DEAL: 650 EUR
  3. Outcome: DEAL at 640 EUR. Last lines: Mediator: Could you both consider 640 EUR for same-day pickup? That’s a reasonable middle  | Buyer: I can do 640 EUR for same-day pickup if we can finalize it today. | Seller: 640 EUR works for me for same-day pickup. Please sen

### Exercise 2 — your analysis

**Comment on the results** :

- First Iteration agrees on weird prices, be it 800 which should not happen or 450 and anything in between (different opening offers)
- With more iterations and more time for the Buyer to learn how the seller sells, he is able to get the seller all the way down to 450
- The speed of the deals slow down with more iterations as the Buyer is trying its best to get to 450 as it starts to learn that is the minimum the seller will go (leked limits)

## Takeaways

1. An LLM agent is the classic loop with a model as the policy.
2. A society is agents + shared board + a protocol (here: round-robin + `DEAL:` stop).
3. Extra sellers change competitive pressure; cross-run memory can change bargaining — measure turns and final price.